In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
   

import torchvision
import torchvision.transforms as transforms

from models import *



def act_quantization(b):

    def uniform_quant(x, b=3):
        xdiv = x.mul(2 ** b - 1)
        xhard = xdiv.round().div(2 ** b - 1)
        return xhard

    class uq(torch.autograd.Function):   # here single underscore means this class is for internal use

        def forward(ctx, input, alpha):
            input_d = input/alpha
            input_c = input_d.clamp(max=1)  # Mingu edited for Alexnet
            input_q = uniform_quant(input_c, b)
            ctx.save_for_backward(input, input_q)
            input_q_out = input_q.mul(alpha)
            return input_q_out

    return uq().apply




def weight_quantization(b):

    def uniform_quant(x, b):
        xdiv = x.mul((2 ** b - 1))
        xhard = xdiv.round().div(2 ** b - 1)  
        return xhard

    class uq(torch.autograd.Function):

        def forward(ctx, input, alpha):
            input_d = input/alpha                          # weights are first divided by alpha                       
            input_c = input_d.clamp(min=-1, max=1)       # then clipped to [-1,1]
            sign = input_c.sign()
            input_abs = input_c.abs()
            input_q = uniform_quant(input_abs, b).mul(sign)
            ctx.save_for_backward(input, input_q)
            input_q_out = input_q.mul(alpha)               # rescale to the original range
            return input_q_out

    return uq().apply



class weight_quantize_fn(nn.Module):
    def __init__(self, w_bit):
        super(weight_quantize_fn, self).__init__()
        self.w_bit = w_bit-1
        self.weight_q = weight_quantization(b=self.w_bit)
        self.wgt_alpha = 0.0

    def forward(self, weight):
        weight_q = self.weight_q(weight, self.wgt_alpha)
        
        return weight_q
    
    

In [ ]:

w_alpha = 4.0  # cliping value
w_bits = 8

x_alpha = 4.0  # clipping value
x_bits = 8

n_batch = 10
n_ch_in = 4
n_ch_out = 8

k_size = 3


x =torch.abs(torch.randn(n_batch,n_ch_in,12,12))  # Note X is unsigned number
act_quant = act_quantization(b= x_bits)  # define activation quantization function
x_quant = act_quant(x, x_alpha)   
x_delta = x_alpha/(2**x_bits-1)   # resolution
x_int   = x_quant/x_delta

# print("X_quant:", x_quant)
# print("resolution:", x_delta)
print("X with integer:", x_int) # why this value is bigger than alpha ?

resolution: 0.01568627450980392
X with integer: tensor([[[[ 45.0000,  47.0000,  46.0000,  ...,  10.0000, 173.0000,  85.0000],
          [ 57.0000,  36.0000,  30.0000,  ...,  66.0000,  17.0000,  54.0000],
          [  5.0000,  57.0000,  10.0000,  ...,  69.0000, 107.0000,   8.0000],
          ...,
          [ 24.0000, 137.0000,  66.0000,  ..., 111.0000,  67.0000, 130.0000],
          [157.0000,   1.0000,   8.0000,  ...,   8.0000,  36.0000,  74.0000],
          [ 90.0000,   3.0000,  88.0000,  ...,  93.0000, 105.0000,  46.0000]],

         [[ 29.0000,  78.0000,  36.0000,  ..., 103.0000, 114.0000,  51.0000],
          [ 20.0000,  31.0000,  76.0000,  ...,  30.0000,  21.0000, 126.0000],
          [ 21.0000,  54.0000,  68.0000,  ...,  37.0000,  58.0000,  90.0000],
          ...,
          [ 26.0000,  49.0000,  33.0000,  ...,  67.0000,  11.0000,  72.0000],
          [ 26.0000,  47.0000,  51.0000,  ...,   3.0000,   3.0000,  78.0000],
          [110.0000,  57.0000, 127.0000,  ..., 131.0000,  67.0

C:\tmp\ipykernel_42524\4243623719.py:37: DeprecationWarning: <class '__main__.act_quantization.<locals>.uq'> should not be instantiated. Methods on autograd functionsare all static, so you should invoke them on the class itself. Instantiating an autograd function will raise an error in a future version of PyTorch.
  return uq().apply


In [12]:

### Initialize conv module
conv = nn.Conv2d(n_ch_in, n_ch_out, kernel_size=k_size, padding=1, bias=False)
conv.weight  = torch.nn.Parameter(torch.randn(n_ch_out,n_ch_in,k_size,k_size))
##########################


weight_quant = weight_quantize_fn(w_bit= w_bits)  ## define quant function
weight_quant.wgt_alpha = torch.tensor(w_alpha)
w_quant      = weight_quant(conv.weight)
w_delta      = w_alpha/(2**(w_bits-1)-1)
w_int        = w_quant/w_delta

print("W with integer:", w_int)

W with integer: tensor([[[[  31.,  -31.,   -1.],
          [  80.,  -13.,    9.],
          [  36.,  -36.,  -46.]],

         [[ -28.,  -28.,  -27.],
          [  43.,  -15.,    1.],
          [  -8.,   18.,  -36.]],

         [[  -4.,   38.,  -15.],
          [   1.,   -9.,   22.],
          [ -23.,   -4.,   18.]],

         [[  28.,  -16.,  -17.],
          [  33.,   22.,   -3.],
          [  46.,  -13.,  -37.]]],


        [[[  -4.,  -52.,   22.],
          [ -53.,   -8.,  -27.],
          [ -11.,   -8.,  -48.]],

         [[  30.,   -0.,    8.],
          [  -3.,  -55.,  -35.],
          [  17.,  -10.,  -57.]],

         [[  34.,   18.,   37.],
          [  44.,   71.,   27.],
          [ -22.,   53.,  -19.]],

         [[  22.,  -29.,    2.],
          [   9.,   72.,   -6.],
          [ -16.,   21.,   -6.]]],


        [[[  58.,   29.,  -20.],
          [ -11.,   22.,   33.],
          [  43.,  -15.,   71.]],

         [[   2.,    4.,  -19.],
          [   6.,   21.,   51.],
     

C:\tmp\ipykernel_42524\4243623719.py:61: DeprecationWarning: <class '__main__.weight_quantization.<locals>.uq'> should not be instantiated. Methods on autograd functionsare all static, so you should invoke them on the class itself. Instantiating an autograd function will raise an error in a future version of PyTorch.
  return uq().apply


In [4]:
###### This cell is the same as red box in the slide #######

conv2 = nn.Conv2d(n_ch_in, n_ch_out, kernel_size=k_size, padding=1, bias=False)
conv2.weight = torch.nn.Parameter(w_int)

psum_int = conv2(x_int)
print("psum int:", psum_int)

psum int: tensor([[[[-4.9510e+03, -1.0064e+04, -6.6110e+03,  ..., -5.8880e+03,
           -1.3176e+04, -6.3820e+03],
          [ 1.9000e+01,  1.1750e+04,  1.2786e+04,  ...,  9.3920e+03,
            1.0062e+04, -1.0741e+04],
          [-1.9220e+03, -1.3544e+04,  5.9900e+03,  ..., -6.9280e+03,
            2.2670e+03, -5.3140e+03],
          ...,
          [ 1.6314e+04, -3.0560e+03, -2.4400e+02,  ...,  7.6110e+03,
           -8.9710e+03, -7.4350e+03],
          [ 1.5277e+04,  5.3440e+03,  3.2170e+03,  ..., -1.8530e+03,
            7.5420e+03,  6.7910e+03],
          [ 3.9280e+03, -3.8420e+03, -2.2310e+03,  ...,  1.9350e+03,
           -1.7230e+03,  1.0040e+03]],

         [[ 5.4540e+03, -1.1804e+04, -1.2710e+03,  ..., -8.0980e+03,
           -4.3570e+03, -8.7570e+03],
          [-3.5900e+02,  1.8610e+03, -1.1280e+03,  ..., -1.3598e+04,
           -3.0590e+03, -1.4800e+02],
          [-8.8280e+03, -6.2450e+03, -5.7860e+03,  ..., -1.4106e+04,
           -7.8430e+03, -7.7290e+03],
          

In [5]:
psum_recovered = psum_int*w_delta*x_delta
print("psum recovered:", psum_recovered)

psum recovered: tensor([[[[-2.4461e+00, -4.9722e+00, -3.2662e+00,  ..., -2.9090e+00,
           -6.5097e+00, -3.1531e+00],
          [ 9.3871e-03,  5.8052e+00,  6.3170e+00,  ...,  4.6402e+00,
            4.9712e+00, -5.3067e+00],
          [-9.4958e-01, -6.6915e+00,  2.9594e+00,  ..., -3.4228e+00,
            1.1200e+00, -2.6254e+00],
          ...,
          [ 8.0600e+00, -1.5098e+00, -1.2055e-01,  ...,  3.7603e+00,
           -4.4322e+00, -3.6733e+00],
          [ 7.5477e+00,  2.6402e+00,  1.5894e+00,  ..., -9.1549e-01,
            3.7262e+00,  3.3551e+00],
          [ 1.9407e+00, -1.8982e+00, -1.1022e+00,  ...,  9.5600e-01,
           -8.5126e-01,  4.9603e-01]],

         [[ 2.6946e+00, -5.8318e+00, -6.2794e-01,  ..., -4.0009e+00,
           -2.1526e+00, -4.3264e+00],
          [-1.7737e-01,  9.1944e-01, -5.5730e-01,  ..., -6.7182e+00,
           -1.5113e+00, -7.3120e-02],
          [-4.3615e+00, -3.0854e+00, -2.8586e+00,  ..., -6.9692e+00,
           -3.8749e+00, -3.8186e+00],
    

In [6]:
#### floating point system result
psum_ref = conv(x)
#################################

print("gap", psum_ref -psum_recovered)

gap tensor([[[[-3.8118e-02, -2.9763e-02, -9.7667e-02,  ..., -2.6651e-02,
           -2.8086e-04,  5.8966e-02],
          [-8.2000e-02, -2.7067e-02, -5.8998e-02,  ..., -8.9088e-02,
           -4.8451e-02, -2.3651e-04],
          [-3.2837e-02, -7.0778e-02,  2.0659e-03,  ..., -1.0060e-02,
           -2.6901e-02, -2.2230e-02],
          ...,
          [-5.3077e-02, -1.8467e-02, -1.6178e-03,  ..., -3.6472e-02,
           -3.4864e-02, -7.1900e-03],
          [ 5.1642e-02, -3.7797e-02, -8.0737e-02,  ..., -7.4770e-02,
           -6.0205e-02, -2.3808e-02],
          [ 4.0670e-02, -4.9476e-02, -7.0851e-02,  ..., -6.7245e-02,
            6.2391e-03, -3.9411e-03]],

         [[-3.3143e-02, -3.6712e-02,  1.6582e-02,  ...,  4.0386e-03,
           -2.4117e-02,  1.9259e-02],
          [-2.2618e-02,  1.2595e-02,  1.3560e-02,  ...,  9.7084e-03,
           -5.4211e-02, -4.8862e-02],
          [-2.8653e-02, -4.9358e-02, -1.9572e-02,  ...,  1.9744e-02,
            1.3736e-02, -2.2550e-02],
          ...,
 